In [22]:
import time
import random
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from dataclasses import dataclass, field

In [23]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [24]:
load_dotenv()
client = genai.Client()

In [25]:
@dataclass
class Retry_Config:
    max_attempts: int = 30
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [26]:
def _handle_retry_or_raise(e, attempt, model_name):
    # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                return

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [27]:
def execute_call(chunk, dimensionality = 768):    
    model_name = "gemini-embedding-2"
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section          
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            #else:
            #    print("Sending initial request...")

            response = client.models.embed_content(
                model = model_name, contents = chunk, config = {"output_dimensionality": dimensionality}
            )

            return response
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue
            

In [42]:
def to_rag_records(result, source_texts, source_id = None):
    records = []
    for i, emb in enumerate(result.embeddings):
        records.append({
            "id": source_id if source_id else i,
            "text": source_texts[i],
            "vector": np.array(emb.values, dtype=np.float32),
            "dim": len(emb.values),
        })
    return records

In [32]:
def load_chunks(format):
    return pd.read_parquet(f"Chunked Data/chunks_{format}.parquet")

In [56]:
def embed_chunks(chunks):
    embedding_result = []
    for idx, chunk in enumerate(chunks["Data_Chunks"].head()):
        embedding_result.append(to_rag_records(execute_call(chunk), [chunk], [idx + 1]))
    return embedding_result

In [33]:
chunks_csv  = load_chunks("csv")
chunks_docx = load_chunks("docx")
chunks_html = load_chunks("html")
chunks_json = load_chunks("json")
chunks_md   = load_chunks("md")
chunks_pdf  = load_chunks("pdf")
chunks_txt  = load_chunks("txt")

In [67]:
embedded_chunks = pd.DataFrame(embed_chunks(chunks_csv))

In [69]:
print(embedded_chunks.to_dict(orient="records"))

[{0: {'id': [1], 'text': 'RowID: 1, DocumentID: DOC-2026-1001, SectionTitle: Deep Dive Analysis Section 1, ChunkSize: 512, Overlap: 50, ContentText: This dummy data row 1 contains representative text for testing CSV tokenizers in retrieval-augmented generation pipelines. Embedding layers process this textual field to yield a 1536-dimensional dense vector representation. Standard configurations require text splitting, and this row acts as a multi-sentence item that should be parsed into individual chunks based on delimiter rules., IsActive: FALSE, SystemScore: 0.911', 'vector': array([-2.24029776e-02,  3.68502140e-02, -1.90687366e-02, -9.37095471e-03,
        1.99079998e-02, -1.42007489e-02, -1.94931086e-02,  5.55868726e-03,
        1.20452968e-02, -8.95151943e-02, -3.81435193e-02,  1.44543434e-02,
        4.74232016e-03,  2.47987863e-02,  1.06882332e-02,  4.44795601e-02,
        3.41591947e-02, -1.14512164e-02, -2.49465052e-02,  6.06420711e-02,
       -2.08469778e-02,  3.55149470e-02, 